# Módulo 3 · Envejecimiento de largo alcance sin foto objetivo
Continuación independiente de **Paper_comparison_bias_v2**. Usa fotos fuente de train de 18–35 años y solicita 60, 65 y 70 años, sin foto real objetivo.

Para cada foto y medida congelada del módulo 1:
\[
g=\phi(I_{generada})-\phi(I_{fuente}),\qquad e_C=g-d_C(a_s,a_t).
\]
Un residuo positivo en una medida orientada al envejecimiento significa **más cambio que el promedio esperado**; uno negativo significa menos. Son unidades estandarizadas del módulo 1, no años ni error contra el futuro real de esa persona.

También compara con FFHQ **transversal** y con el contraste latino **transversal**, separados de la referencia colombiana **longitudinal**. “Sin editar” tiene cambio cero y residuo \(-d_C\). No se inventa un oracle ni una foto vieja. Los saltos grandes pueden ayudar a identificar π, pero se mantienen los controles de rango, correlación, condición y dosis.

**Salidas separadas:** panel y auditoría, imágenes por modelo, reanudación, métricas por foto/medida, intervalos por identidad, contrastes pareados, cobertura, fallos, figuras y muestras visuales. No modifica ni ajusta el módulo 1.

## 1. Colab y repositorio
Ejecuta desde la versión actualizada del repositorio que contiene este notebook. Si ya estás dentro del proyecto, la celda conserva esa ruta.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, gc

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

REPO_DIR = Path.cwd()
if not (REPO_DIR / "src/comparisong/long_gap_panel.py").is_file():
    REPO_DIR = Path("/content/longitudinal-aging-diffusion")
if not (REPO_DIR / "src/comparisong/long_gap_panel.py").is_file():
    raise FileNotFoundError("Actualiza el repositorio con este módulo y ajusta REPO_DIR antes de continuar.")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repositorio:", REPO_DIR)

In [ ]:
%pip install -e ".[aging-bias,evaluation,comparison,auxiliary]"

## 2. Única configuración de rutas y modelos
Cambia `MODELS` libremente. SAM/FRAN usan lotes; HRFAE agrupa por edad y ajusta el lote a la VRAM; FADING reutiliza la inversión por fuente. Nuestros modelos directos y picking conservan su inferencia por transición. La selección usa exclusivamente train; las otras carpetas solo comprueban qué edades ya existen.

Las referencias permanecen congeladas. El solapamiento con módulo 1 se registra: este experimento con fuentes de train no demuestra generalización a identidades independientes.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from src.aging_bias.references import ReferenceBundle
from src.aging_bias.config import FeatureConfig, PaperV2Config
from src.comparisong.long_gap_panel import LongGapPanelConfig, select_long_gap_panel
from src.comparisong.long_gap_evaluation import LongGapEvaluationConfig, evaluate_long_gap
from src.comparisong.identity_metadata import load_gender_csv, TEST_ID_GENDERS

TRAIN_ROOT = Path("/content/drive/MyDrive/dataset_def/train")
# Incluye todas las carpetas conocidas: una edad cercana fuera de train también excluye la persona.
OBSERVED_ROOTS = [
    TRAIN_ROOT,
    Path("/content/drive/MyDrive/dataset_def/val"),
    Path("/content/drive/MyDrive/dataset_def/test"),
    Path("/content/drive/MyDrive/dataset_unificado"),
    Path("/content/drive/MyDrive/daytaset_unificado_test"),
]
REFERENCE_DIR = Path("/content/drive/MyDrive/aging_bias_references_v2")
CHECKPOINT_PATH = Path("/content/drive/MyDrive/adapter_inference.pt")
OUTPUT = Path("/content/drive/MyDrive/paper_long_gap_counterfactual_v1")
ASSETS = Path("/content/evaluation_models_v2")
BENCHMARK_ROOT = Path("/content/aging_benchmarks")
GENDER_CSV = Path("/content/genero_por_id.csv")  # Opcional si el diccionario cubre los IDs.
GENDER_BY_ID = {}  # Etiquetas conocidas adicionales: {"id_123": "female"}.
FADING_SPECIALIZED_PATH = globals().get("FADING_SPECIALIZED_PATH")

MODELS = ["SAM", "FRAN", "FADING", "HRFAE",
          "ours_strength_0.31", "ours_picking_no_prompt", "ours_picking_prompt"]
N_TRANSITIONS = 350
SOURCE_AGES = (18, 35)
TARGET_AGES = (60, 65, 70)
EXCLUSION_MARGIN = 10  # Confirmado: excluye edades observadas ±10 de CUALQUIER objetivo.
MIN_GAP = 25
SEED = 2026
BATCH_SIZE = 32
PROGRESS_EVERY = 100
N_BOOT = 1000
RUN_DINO = True  # Robustez con PCA/referencias ya congeladas en módulo 1.
ALLOW_DOWNLOAD = True
EXCLUDE_UNPARSED_IDENTITIES = True

if not MODELS or len(MODELS) != len(set(MODELS)):
    raise ValueError("MODELS debe contener nombres distintos y al menos un modelo.")
if OUTPUT.resolve() == REFERENCE_DIR.resolve() or REFERENCE_DIR.resolve() in OUTPUT.resolve().parents:
    raise ValueError("El módulo 1 es de solo lectura; cambia OUTPUT.")
OUTPUT.mkdir(parents=True, exist_ok=True)
references = ReferenceBundle.load(REFERENCE_DIR, load_training_data=False)
if not any(a.startswith("zone_") for a in references.attributes):
    raise ValueError("Carga el bundle del módulo 1 con sus índices regionales congelados.")
frozen_features = dict(references.metadata["config"]["features"])
frozen_features["new_attributes"] = tuple(frozen_features.get("new_attributes", ()))
features = FeatureConfig(**frozen_features)
generated_roots = {m: OUTPUT / "generated" / m for m in MODELS}
if RUN_DINO:
    from src.comparisong.frozen_dino import validate_frozen_dino_files
    validate_frozen_dino_files(REFERENCE_DIR)
print("Modelos:", MODELS)
print("Medidas congeladas:", references.attributes)
print("Soporte global:", references.support, "| edades representativas FFHQ:", references.p_ages.tolist())
print("La inferencia y las métricas guardan todo bajo:", OUTPUT)

## 3. Inventario completo y exclusión de edades observadas
Se inventarían ambas edades de los pares si entregas manifiestos; aquí se recorre una vez cada carpeta, sin leer píxeles. El parser del proyecto reconoce `edad[_índice].jpg/png/jpeg/webp`. Una persona con fotos de edad desconocida o subcarpetas queda excluida por defecto. Revisa las carpetas faltantes: la exclusión solo puede garantizarse respecto del inventario suministrado.

In [ ]:
from data.indexing import build_image_manifest

genders = dict(TEST_ID_GENDERS)
if GENDER_CSV.is_file():
    genders.update(load_gender_csv(GENDER_CSV))
genders.update(GENDER_BY_ID)
inventory_dir = OUTPUT / "panel" / "input_inventories"
inventory_dir.mkdir(parents=True, exist_ok=True)
inventory_paths, scan_audits, unknown_age_ids = [], {}, set()
seen_roots = set()
train_inventory = None
for root in OBSERVED_ROOTS:
    root = Path(root)
    key = str(root.resolve())
    if key in seen_roots:
        continue
    seen_roots.add(key)
    if not root.is_dir():
        if root.resolve() == TRAIN_ROOT.resolve():
            raise FileNotFoundError(f"No existe TRAIN_ROOT: {root}")
        scan_audits[key] = {"missing_root": True}
        print("[inventario] carpeta no disponible:", root)
        continue
    records, audit = build_image_manifest(root, use_cached_manifest=False,
        min_age=0, max_age=120, normalize_over_100=False, gender_by_person=genders)
    scan_audits[key] = audit
    for item in audit["skipped_files"] + audit["filtered_ages"]:
        relative = Path(item["relative_path"])
        if relative.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}:
            unknown_age_ids.add(relative.parts[0])
    for identity in root.iterdir():
        if identity.is_dir() and any(p.is_dir() for p in identity.iterdir()):
            unknown_age_ids.add(identity.name)
    frame = pd.DataFrame([dict(identity_id=r.person_id, age=r.age,
        path=str(r.path(root)), gender=r.gender or "") for r in records],
        columns=["identity_id", "age", "path", "gender"])
    name = hashlib.sha256(key.encode()).hexdigest()[:12] + ".csv"
    path = inventory_dir / name
    frame.to_csv(path, index=False)
    if root.resolve() == TRAIN_ROOT.resolve():
        train_inventory = path
    else:
        inventory_paths.append(path)
    print(f"[inventario] {root}: {len(frame)} fotos / {frame.identity_id.nunique()} identidades")
(inventory_dir / "scan_audits.json").write_text(json.dumps(scan_audits, indent=2, default=str))
(inventory_dir / "unknown_age_identities.json").write_text(json.dumps(sorted(unknown_age_ids), indent=2))
print("Identidades con edad no verificable:", len(unknown_age_ids))
assert train_inventory is not None

In [ ]:
panel_config = LongGapPanelConfig(
    train_manifest_path=train_inventory,
    inventory_manifest_paths=tuple(inventory_paths),
    output_dir=OUTPUT / "panel",
    n_transitions=N_TRANSITIONS,
    source_age_min=SOURCE_AGES[0], source_age_max=SOURCE_AGES[1],
    target_ages=TARGET_AGES, exclusion_margin=EXCLUSION_MARGIN, min_gap=MIN_GAP,
    seed=SEED, reference_bundle_path=REFERENCE_DIR,
    excluded_identity_ids=tuple(sorted(unknown_age_ids)) if EXCLUDE_UNPARSED_IDENTITIES else (),
)
selected = select_long_gap_panel(panel_config)
panel = selected["panel"]
PANEL_PATH = selected["paths"]["panel"]
print(json.dumps(selected["diagnostics"], indent=2))
display(selected["exclusions"].groupby("reason").size().rename("identidades").to_frame())
display(panel.groupby("target_age").agg(transiciones=("sample_id","size"),
    personas=("identity_id","nunique"), salto_min=("delta_age","min"), salto_max=("delta_age","max"),
    distancia_min_a_foto_observada=("nearest_observed_target_age_distance","min")))
display(panel.head(12))
assert len(panel) == N_TRANSITIONS and panel.target_path.eq("").all()
assert panel.groupby("identity_id").source_path.nunique().max() == 1
# Una foto observada cerca de cualquier objetivo invalida la identidad entera.
observed = selected["inventory"]
near = observed.age.map(lambda age: any(abs(age-t) <= EXCLUSION_MARGIN for t in TARGET_AGES))
assert not set(panel.identity_id) & set(observed.loc[near, "identity_id"])
supported = ~references.extrapolated(panel.source_age, panel.target_age)
adult_min = references.metadata.get("reliability_policy", {}).get("retest_min_age") or 0
supported &= panel.source_age.ge(adult_min).to_numpy()
support_table = panel.assign(reference_supported=supported).groupby("target_age").reference_supported.agg(["sum","count"])
display(support_table.rename(columns={"sum":"con_soporte","count":"solicitadas"}))
if not supported.any():
    raise ValueError("Ninguna transición tiene soporte congelado: revisa las referencias antes de generar.")
if not supported.all():
    print("AVISO: las transiciones sin soporte quedarán fuera del resumen regional primario.")
for filename in ("reference_quality.csv", "population_contrasts.csv"):
    path = REFERENCE_DIR / filename
    if path.is_file():
        print("Diagnóstico original del módulo 1:", filename)
        display(pd.read_csv(path).head(30))

## 4. Preflight de generación
FADING y Cradle2Cane requieren género conocido. Si falta, completa el CSV guardado y actualiza `GENDER_CSV`; no se adivinan etiquetas. MiVOLO se instala con el arreglo de setuptools usado antes. La evaluación independiente usa FairFace/AdaFace, no los críticos de picking.

In [ ]:
if any(m in {"FADING", "Cradle2Cane"} for m in MODELS):
    missing = sorted(set(panel.identity_id) - set(genders))
    if missing:
        path = OUTPUT / "panel" / "missing_gender.csv"
        pd.DataFrame({"identity_id":missing, "gender":[""]*len(missing)}).to_csv(path,index=False)
        raise ValueError(f"Falta género en {len(missing)} identidades. Completa {path}, ajusta GENDER_CSV y repite inventario/panel.")
if any(m.startswith("ours_") for m in MODELS) and not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(CHECKPOINT_PATH)
if any(m.startswith("ours_picking_") for m in MODELS):
    def pip_install(*args):
        subprocess.check_call([sys.executable, "-m", "pip", "install", *args])
    pip_install("setuptools==80.9.0", "wheel", "timm>=1.0.11,<2")
    pip_install("--no-build-isolation", "--no-deps",
        "git+https://github.com/WildChlamydia/MiVOLO.git@37475e3f8818b5f22448003feec3e64b01bfb188")
print("Preflight de fuentes, objetivos y etiquetas completado.")

## 5. Generar o reanudar las mismas 350 transiciones
Esta fase permite objetivos sin imagen real. `overwrite=False` y validación rápida recuperan las imágenes ya hechas. Mantén la misma selección/configuración para reanudar después de cambiar GPU; usa otro OUTPUT si cambias el experimento.

In [ ]:
from src.comparisong.generate_paper_predictions import generate_paper_predictions

generation_config = PaperV2Config(
    test_manifest_path=PANEL_PATH, source_root=None,
    generated_roots=generated_roots, models=MODELS,
    reference_bundle_path=REFERENCE_DIR, output_dir=OUTPUT / "generation_run",
    features=features, include_dex_appendix=False, include_kid=False,
    resume=True, batch_size=BATCH_SIZE, seed=SEED,
)
generated = generate_paper_predictions(
    generation_config, checkpoint_path=CHECKPOINT_PATH,
    model_loader_config={"root":str(BENCHMARK_ROOT),
        "fading_specialized_path":FADING_SPECIALIZED_PATH},
    gender_csv_path=str(GENDER_CSV) if GENDER_CSV.is_file() else None,
    gender_by_id=genders, fading_group_targets=True,
    inference_config={
        "image_size":400, "num_inference_steps":37, "text_reference_mode":"source_age",
        "age_guidance_scale":2.0, "text_guidance_scale":7.0, "image_guidance_scale":1.5,
    },
    picking_options={
        "delta_age_strength_map":{0:.04, 10:.15, 20:.22, 30:.23, 40:.29, 60:.34},
        "max_trials_per_target":10, "min_strength":.04, "max_strength":.45,
        "overwrite":False,
    },
    batch_size=BATCH_SIZE, progress_every=PROGRESS_EVERY,
    resume_validation="fast", resume_workers=16, local_files_only=False,
)
display(pd.crosstab(generated["manifest"].model_name, generated["manifest"].inference_status))
generation_failures = generated["manifest"].query("inference_status != 'ok'")
display(generation_failures)
gc.collect()
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 6. Evaluación regional congelada y métricas independientes
No llama al evaluador de pares reales. Usa el manifiesto contrafactual original: la edad objetivo es una solicitud y `target_path` permanece vacío. El modelo “sin editar” se añade automáticamente. Los resúmenes y contrastes usan las mismas transiciones válidas para todos los modelos, con medias por identidad e intervalos bootstrap por identidad.

In [ ]:
from src.comparisong.evaluation_backends_v2 import prepare_paper_bundle

# Reubicar el archivo de landmarks conserva el hash solo si sus bytes son los mismos.
features.landmark_model_path = str(ASSETS / "face_landmarker.task")
evaluation_bundle = prepare_paper_bundle(
    ASSETS, feature_config=features, allow_download=ALLOW_DOWNLOAD,
    include_dex=False, include_kid=False,
)
evaluation_config = LongGapEvaluationConfig(
    manifest_path=PANEL_PATH, generated_roots=generated_roots, models=MODELS,
    reference_bundle_path=REFERENCE_DIR, output_dir=OUTPUT / "metrics",
    cache_dir=OUTPUT / "cache", n_boot=N_BOOT, seed=SEED, batch_size=16,
)
result = evaluate_long_gap(evaluation_config, bundle=evaluation_bundle, reference_bundle=references)

## 7. Diagnóstico completo: cobertura, signos, semejanza a FFHQ y π
El error regional es respecto de una trayectoria promedio, no del futuro individual. “Más que lo esperado” y “menos que lo esperado” solo se interpretan así en medidas orientadas. Los errores de edad independiente se expresan en años. La identidad se compara con la fuente. No hay similitud con un objetivo real ni KID contra fotos futuras inexistentes.

In [ ]:
metrics_dir = Path(result["output_dir"])
figures = OUTPUT / "figures"
figures.mkdir(exist_ok=True)
print(json.dumps(result["completeness"], indent=2))
print("\nCOBERTURA (fallos y soporte nunca se convierten en ceros)")
display(result["coverage"])
print("\nFALLOS")
display(result["failures"])
all_summary = result["summary"].query("group == 'all'")
print("\nRESUMEN GLOBAL: todas las medidas y métricas")
with pd.option_context("display.max_rows", None):
    display(all_summary)
print("\nEDADES OBJETIVO: errores regionales por modelo")
display(result["summary"].query("group == 'target' and metric in ['error','absolute_error','rmse']"))
print("\nDIAGNÓSTICO ESTRUCTURAL CONJUNTO: π no se fuerza si falla identificación")
with pd.option_context("display.max_columns", None):
    display(result["structural_bias"])
print("\nDIFERENCIAS PAREADAS DEL ERROR ABSOLUTO (modelo_a - modelo_b; negativo favorece a A)")
display(result["paired"].query("group == 'all' and metric in ['absolute_error','age_mae_indep','adaface_id_source']"))

per_measure = result["per_measure"]
print("Motivos de exclusi"+chr(243)+"n de medidas:")
display(per_measure.groupby(["model_name","missing_reason"]).size().unstack(fill_value=0))
signs = per_measure.query("valid and aging_oriented").copy()
sign_counts = signs.groupby(["model_name","measure","interpretation"]).size().unstack(fill_value=0)
sign_counts.to_csv(metrics_dir / "sign_counts.csv")
print("\nSIGNOS POR FOTO Y ZONA: recuentos")
display(sign_counts)
sign_map = per_measure.pivot(index=["sample_id","identity_id","source_age","target_age","model_name"],
    columns="measure", values="error")
sign_map.to_csv(metrics_dir / "signed_residuals_by_photo.csv")
print("Mapa completo de signos:", metrics_dir / "signed_residuals_by_photo.csv")
display(sign_map.head(24))
regional = per_measure.query("valid and in_common_panel and measure_kind == 'zone'").copy()
if not regional.empty:
    proximity = regional.groupby(["model_name","measure","identity_id"])[
        ["absolute_error","ffhq_absolute_error","latino_cross_absolute_error"]].mean().groupby(level=["model_name","measure"]).mean()
    # Medias por identidad sobre el panel común; los CI comparables están en summary.csv (medias por identidad).
    proximity.to_csv(metrics_dir / "reference_proximity_descriptive.csv")
    print("\nDISTANCIA AL CAMBIO ESPERADO: menor es más próximo a esa referencia")
    display(proximity)
print("Todos los CSV y su configuración:", metrics_dir)

In [ ]:
import matplotlib.pyplot as plt

regional_summary = all_summary[
    all_summary.measure.str.startswith("zone_") & all_summary.metric.eq("error")
]
zones = list(regional_summary.measure.unique())
if zones:
    fig, axes = plt.subplots(1, len(zones), figsize=(6*len(zones), 5), squeeze=False)
    for ax, zone in zip(axes[0], zones):
        frame = regional_summary[regional_summary.measure.eq(zone)].set_index("model_name")
        frame = frame.reindex([m for m in MODELS+["source_noedit"] if m in frame.index])
        x = np.arange(len(frame))
        ax.scatter(x, frame.value, color="tab:blue")
        for i, row in enumerate(frame.itertuples()):
            if np.isfinite([row.ci_low,row.ci_high]).all():
                ax.plot([i,i], [row.ci_low,row.ci_high], color="tab:blue")
        ax.axhline(0, color="black", linewidth=1)
        ax.set_xticks(x, frame.index, rotation=65, ha="right")
        ax.set_title(zone)
        ax.set_ylabel("Cambio generado − esperado latino (unidades congeladas)")
    fig.tight_layout()
    fig.savefig(figures / "signed_regional_residuals.png", dpi=180, bbox_inches="tight")
    plt.show()

age_summary = all_summary.query("measure == 'operational' and metric == 'age_mae_indep'")
if not age_summary.empty:
    fig, ax = plt.subplots(figsize=(9,4))
    ax.bar(age_summary.model_name, age_summary.value)
    ax.set_ylabel("MAE de edad independiente vs edad solicitada (años)")
    ax.tick_params(axis="x", rotation=60)
    fig.tight_layout()
    fig.savefig(figures / "independent_age_mae.png", dpi=180, bbox_inches="tight")
    plt.show()

## 8. Inspección visual con signos por foto
Fuentes elegidas sin mirar qué modelo gana. Se guardan las láminas y los signos de todos los modelos para esas transiciones. El CSV anterior conserva todas las fotos, no solo esta muestra.

In [ ]:
from PIL import Image
N_DIAGNOSTIC_IMAGES = 6
sample_rows = panel.sample(n=min(N_DIAGNOSTIC_IMAGES,len(panel)), random_state=SEED)
for sample in sample_rows.to_dict("records"):
    names = ["source_noedit"] + MODELS
    fig, axes = plt.subplots(1,len(names),figsize=(3*len(names),3.8),squeeze=False)
    for ax, name in zip(axes[0],names):
        path = Path(sample["source_path"]) if name=="source_noedit" else (
            generated_roots[name] / sample["identity_id"] / (sample["sample_id"]+".png"))
        if path.is_file():
            with Image.open(path) as im:
                ax.imshow(im.convert("RGB"))
        ax.set_title(name, fontsize=9)
        ax.axis("off")
    fig.suptitle(f'{sample["identity_id"]}: {sample["source_age"]:.0f} → {sample["target_age"]:.0f} | sin objetivo real')
    fig.tight_layout()
    fig.savefig(figures / (sample["sample_id"]+".png"), dpi=130, bbox_inches="tight")
    plt.show()
    display(per_measure[
        per_measure.sample_id.eq(sample["sample_id"]) & per_measure.measure_kind.eq("zone")
    ][["model_name","measure","model_change","expected_change","error","interpretation","valid"]])

## 9. Robustez DINO congelada (opcional)
Reutiliza el encoder, PCA y referencias del módulo 1: no vuelve a ajustar PCA ni trayectorias. Sus coordenadas latentes no son zonas anatómicas y sus signos no se interpretan como “más arrugas”. Se escribe en otra carpeta y conserva la misma selección/modelos.

In [ ]:
if RUN_DINO:
    from src.comparisong.frozen_dino import load_frozen_dino_backend
    from dataclasses import replace
    dino_reference, dino_bundle = load_frozen_dino_backend(
        REFERENCE_DIR, root=ASSETS, landmark_backend=evaluation_bundle["landmark_backend"],
        cache_dir=OUTPUT / "cache", allow_download=ALLOW_DOWNLOAD, device="auto",
        landmark_model_path=evaluation_bundle["feature_config"].landmark_model_path,
    )
    try:
        dino_config = replace(evaluation_config,
            reference_bundle_path=REFERENCE_DIR / "dinov2",
            output_dir=OUTPUT / "metrics_dinov2", cache_dir=OUTPUT / "cache_dinov2")
        dino_result = evaluate_long_gap(dino_config,
            bundle={**evaluation_bundle, **dino_bundle}, reference_bundle=dino_reference)
        display(dino_result["structural_bias"])
        display(dino_result["summary"].query("group == 'all'"))
    finally:
        encoder = getattr(dino_bundle.get("feature_backend"), "encoder", None)
        if hasattr(encoder,"to"):
            encoder.to("cpu")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("DINO desactivado explícitamente; el análisis primario queda guardado.")

## 10. Registro final
Los intervalos condicionan a las referencias congeladas: no incorporan su incertidumbre de ajuste. FFHQ es una comparación transversal y el experimento usa fuentes de train. Las edades excluidas y el solapamiento con módulo 1 quedan en la auditoría; las edades no vistas se garantizan únicamente frente a las fotos inventariadas.

In [ ]:
final_record = {
    "protocol":"long-gap-counterfactual-frozen-module1-v1",
    "panel_path":str(PANEL_PATH), "output":str(OUTPUT),
    "models":MODELS, "n_transitions":len(panel), "no_real_target":True,
    "selection":selected["diagnostics"],
    "evaluation":result["completeness"],
    "dino_requested":RUN_DINO,
    "dino_complete":bool(RUN_DINO and "dino_result" in globals()),
    "main_tables":[str(metrics_dir / (name+".csv")) for name in
        ["per_sample","per_measure","summary","paired","structural_bias","coverage","failures"]],
}
(OUTPUT / "experiment_summary.json").write_text(json.dumps(final_record,indent=2,default=str))
print(json.dumps(final_record,indent=2,default=str))